# Netflix Recommendation System

# Notebook 5: Recommendation System

## Author
Bishal Gurung

## Objective

The goal of this notebook is to build a recommendation system that suggests movies or TV shows based on content similarity and user preferences.

### Techniques Used

- Content-Based Filtering
- TF-IDF Vectorization
- Cosine Similarity
- User-Item Matrix
- Matrix Factorization (NMF)

# Import Required Libraries

In [3]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import NMF

import joblib

# Load Cleaned Dataset

In [4]:
df = pd.read_csv("../data/processed/netflix_cleaned.csv")

df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,year_added,month_added
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",2021,9
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",2021,9
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,2021,9
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",2021,9
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,2021,9


# Content-Based Filtering

Content-based recommendation suggests titles that are similar based on their metadata, such as genres and descriptions. For this project, we combine genre and description into a single text feature.

In [5]:
# Create recommendation text feature

df["content"] = (
    df["listed_in"].fillna("")
    + " "
    + df["description"].fillna("")
)

df[["title", "content"]].head()

,title,content
0,Dick Johnson Is Dead,Documentaries As her father nears the end of h...
1,Blood & Water,"International TV Shows, TV Dramas, TV Mysterie..."
2,Ganglands,"Crime TV Shows, International TV Shows, TV Act..."
3,Jailbirds New Orleans,"Docuseries, Reality TV Feuds, flirtations and ..."
4,Kota Factory,"International TV Shows, Romantic TV Shows, TV ..."


In [6]:
from sklearn.metrics.pairwise import cosine_similarity

tfidf_rec = TfidfVectorizer(
    stop_words="english",
    max_features=10000
)

tfidf_matrix = tfidf_rec.fit_transform(df["content"])
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

indices = pd.Series(df.index, index=df["title"]).drop_duplicates()

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)
print("Cosine Similarity Shape:", cosine_sim.shape)

TF-IDF Matrix Shape: (8807, 10000)
Cosine Similarity Shape: (8807, 8807)


In [7]:
def recommend_movies(title, n=10):
    """
    Recommend movies/shows similar to the selected title.
    """
    if title not in indices:
        return "Title not found in dataset."
    
    idx = indices[title]

    similarity_scores = list(enumerate(cosine_sim[idx]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    similarity_scores = similarity_scores[1:n+1]

    movie_indices = [i[0] for i in similarity_scores]

    return df.iloc[movie_indices][["title", "type", "listed_in", "description"]]

In [8]:
recommend_movies("Kota Factory")

,title,type,listed_in,description
3461,Cheese in the Trap,TV Show,"International TV Shows, Korean TV Shows, Roman...","In this adaptation of a popular webtoon, a poo..."
7632,"O-Negative, Love Can’t Be Designed",TV Show,"International TV Shows, Romantic TV Shows, TV ...",Five schoolmates who share a blood type naviga...
4265,Single Ladies Senior,TV Show,"International TV Shows, Romantic TV Shows, TV ...",Four best friends and spirited career women na...
2362,The Politician,TV Show,"TV Comedies, TV Dramas, Teen TV Shows",Rich kid Payton has always known he's going to...
8334,The Great Train Robbery,TV Show,"British TV Shows, Crime TV Shows, Internationa...",This two-part tale delivers the true story of ...
397,Feels Like Ishq,TV Show,"International TV Shows, Romantic TV Shows, TV ...",Short films follow young adults as they naviga...
805,Racket Boys,TV Show,"International TV Shows, TV Comedies, TV Dramas",A city kid is brought to the countryside by hi...
8165,Teresa,TV Show,"International TV Shows, Romantic TV Shows, Spa...","We all want so much more than we have, but how..."
3742,Somewhere Only We Know,TV Show,"International TV Shows, Romantic TV Shows, TV ...",A language major bickers with – and falls for ...
3845,Cinta 100KG,TV Show,"International TV Shows, Romantic TV Shows, TV ...","Two female friends, each with confidence issue..."


**Insight:** The recommender finds titles with similar genre and description patterns.

# User-Item Matrix

The original dataset has no user ratings. To learn collaborative filtering, we create synthetic users and ratings.

In [9]:
np.random.seed(42)

num_users = 500
num_movies = len(df)
ratings_list = []

# Each synthetic user rates around 30 random movies/shows
for user_id in range(1, num_users + 1):
    rated_movies = np.random.choice(df.index, size=30, replace=False)
    
    for movie_id in rated_movies:
        rating = np.random.choice([1, 2, 3, 4, 5], p=[0.05, 0.10, 0.20, 0.35, 0.30])
        ratings_list.append([user_id, movie_id, rating])

ratings_df = pd.DataFrame(
    ratings_list,
    columns=["user_id", "movie_id", "rating"]
)

ratings_df.head()

,user_id,movie_id,rating
0,1,4970,4
1,1,3362,3
2,1,5494,4
3,1,1688,4
4,1,1349,5


In [10]:
user_item_matrix = ratings_df.pivot_table(
    index="user_id",
    columns="movie_id",
    values="rating",
    fill_value=0
)

print("User-Item Matrix Shape:", user_item_matrix.shape)
user_item_matrix.head()

User-Item Matrix Shape: (500, 7242)


movie_id,0,1,3,4,6,7,9,10,13,14,...,8792,8793,8795,8796,8797,8798,8799,8803,8804,8806
user_id,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


**Meaning:** Rows are users, columns are movie IDs, and values are ratings.

# Matrix Factorization using NMF

Matrix Factorization breaks the large user-item matrix into smaller hidden feature matrices.

In [11]:
from sklearn.decomposition import NMF

nmf_model = NMF(
    n_components=20,
    random_state=42,
    max_iter=500
)

user_features = nmf_model.fit_transform(user_item_matrix)
movie_features = nmf_model.components_

print("User features shape:", user_features.shape)
print("Movie features shape:", movie_features.shape)

User features shape: (500, 20)
Movie features shape: (20, 7242)


c:\Users\acer\OneDrive\Desktop\netflix_recommendation_system\venv\Lib\site-packages\sklearn\decomposition\_nmf.py:1723: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(


In [12]:
predicted_ratings = np.dot(user_features, movie_features)

predicted_ratings_df = pd.DataFrame(
    predicted_ratings,
    index=user_item_matrix.index,
    columns=user_item_matrix.columns
)

predicted_ratings_df.head()

movie_id,0,1,3,4,6,7,9,10,13,14,...,8792,8793,8795,8796,8797,8798,8799,8803,8804,8806
user_id,,,,,,,,,,,,,,,,,,,,,
1,0.009367,0.000000,0.003058,0.000691,0.030671,0.001102,0.003523,0.001358,0.017856,0.002981,...,0.007245,0.023022,0.052341,0.007891,0.013080,0.002263,0.009924,0.002246,0.005616,0.018386
2,0.000000,0.000000,0.002219,0.000000,0.000000,0.000000,0.001455,0.000000,0.000000,0.000000,...,0.003022,0.000156,0.495102,0.010273,0.000000,0.000000,0.052435,0.011850,0.000000,0.091764
3,0.127895,0.191043,0.113900,0.755096,0.000090,0.002833,0.000000,0.000000,0.140225,0.000000,...,0.000029,0.003679,0.000348,0.000000,0.015464,0.008979,0.009921,0.007481,0.042837,0.099188
4,0.006305,0.000000,0.000000,0.036469,0.006756,0.003429,0.016759,0.000000,0.000000,0.000000,...,0.013783,0.029154,0.411027,0.016512,0.004477,0.000000,0.087623,0.025274,0.009521,0.006330
5,0.049364,0.043692,0.021487,0.141994,0.007575,0.012025,0.000694,0.004380,0.032550,0.000468,...,0.002474,0.007039,0.053439,0.001635,0.018768,0.004942,0.002911,0.003528,0.009616,0.035737


# Personalized Recommendations from Matrix Factorization

In [13]:
def recommend_for_user(user_id, n=10):
    """
    Recommend movies for an existing synthetic user.
    """
    user_predictions = predicted_ratings_df.loc[user_id]

    already_rated = ratings_df[
        ratings_df["user_id"] == user_id
    ]["movie_id"].values

    user_predictions = user_predictions.drop(labels=already_rated, errors="ignore")

    top_movie_ids = user_predictions.sort_values(ascending=False).head(n).index

    return df.iloc[top_movie_ids][["title", "type", "listed_in", "description"]]

In [14]:
recommend_for_user(1, n=10)

,title,type,listed_in,description
7403,Marauders,Movie,Action & Adventure,A series of high-stakes thefts at banks owned ...
7536,My Ex-Ex,Movie,"Comedies, Romantic Movies",A recently dumped attorney hires a psychic to ...
4303,Vanjagar Ulagam,Movie,"Dramas, International Movies, Thrillers",Shyam wakes to discover he's suspected in a mu...
2031,Cuties,Movie,"Dramas, International Movies",Eleven-year-old Amy starts to rebel against he...
8529,The Tiger Hunter,Movie,"Comedies, Dramas, Independent Movies","It's 1979, and engineer Sami is excited to lea..."
2548,Trial By Media,TV Show,"Crime TV Shows, Docuseries","In this true crime docuseries, some of the mos..."
2928,Good Time,Movie,"Dramas, Independent Movies, Thrillers","After spearheading an ill-fated bank robbery, ..."
3413,Verses of Love 2,Movie,"Dramas, Faith & Spirituality, International Mo...","Now a lecturer in Edinburgh, Fahri tries to be..."
3253,"Lorena, Light-Footed Woman",Movie,"Documentaries, International Movies, Sports Mo...",Lorena Ramírez of Mexico's Rarámuri community ...
2648,The House of Flowers,TV Show,"International TV Shows, Spanish-Language TV Sh...","In this dark comedy, a wealthy matriarch tries..."


# New User Recommendation System

This section is useful for the Streamlit app. A new user rates 2–4 titles, and the system recommends similar titles.

In [15]:
def recommend_for_new_user(user_ratings, n=10):
    """
    user_ratings should be a dictionary.
    Example:
    {
        "Kota Factory": 5,
        "Blood & Water": 4,
        "Ganglands": 3
    }
    """
    liked_movie_indices = []

    for title, rating in user_ratings.items():
        if title in indices:
            liked_movie_indices.append(indices[title])
        else:
            print(f"{title} not found in dataset")

    if len(liked_movie_indices) == 0:
        return pd.DataFrame()

    similarity_scores = cosine_sim[liked_movie_indices]
    avg_similarity_scores = similarity_scores.mean(axis=0)

    scores = pd.Series(avg_similarity_scores, index=df.index)
    scores = scores.drop(labels=liked_movie_indices, errors="ignore")

    top_indices = scores.sort_values(ascending=False).head(n).index

    return df.loc[top_indices, ["title", "type", "listed_in", "description"]]

In [16]:
new_user = {
    "Kota Factory": 5,
    "Blood & Water": 4,
    "Ganglands": 3
}

recommend_for_new_user(new_user, n=10)

,title,type,listed_in,description
8165,Teresa,TV Show,"International TV Shows, Romantic TV Shows, Spa...","We all want so much more than we have, but how..."
3177,Triad Princess,TV Show,"Crime TV Shows, International TV Shows, Romant...",After jostling her way into a gig as a celebri...
7463,Miss Dynamite,TV Show,"Crime TV Shows, International TV Shows, Spanis...","Wealthy, beautiful Valentina falls in love, on..."
3976,The Eagle of El-Se'eed,TV Show,"Crime TV Shows, International TV Shows, TV Act...",A police officer and a drug lord become embroi...
7632,"O-Negative, Love Can’t Be Designed",TV Show,"International TV Shows, Romantic TV Shows, TV ...",Five schoolmates who share a blood type naviga...
2184,Get Even,TV Show,"British TV Shows, Crime TV Shows, Internationa...","In a secret act of skillful revenge, four priv..."
4271,Lion Pride,TV Show,"International TV Shows, Romantic TV Shows, TV ...","After crossing paths at a crime scene, a renow..."
2606,Extracurricular,TV Show,"Crime TV Shows, International TV Shows, Korean...",A model high school student who's steeped in a...
4487,Accidentally in Love,TV Show,"International TV Shows, Romantic TV Shows, TV ...","Rejecting the demands of her wealthy family, a..."
8334,The Great Train Robbery,TV Show,"British TV Shows, Crime TV Shows, Internationa...",This two-part tale delivers the true story of ...


In [19]:
joblib.dump(
    tfidf_rec,
    "../models/6.Final_Model/tfidf_vectorizer.pkl"
)

joblib.dump(
    cosine_sim,
    "../models/6.Final_Model/cosine_similarity.pkl"
)

joblib.dump(
    nmf_model,
    "../models/6.Final_Model/nmf.pkl"
)
print("All models saved successfully!")

All models saved successfully!


# Conclusion

In this notebook, we built a movie recommendation system using multiple recommendation techniques.

### Techniques Implemented

- Content-Based Filtering
- TF-IDF Vectorization
- Cosine Similarity
- User-Item Matrix
- Matrix Factorization (NMF)

The recommendation artifacts were saved and later integrated into the Streamlit web application, allowing users to rate movies and receive personalized recommendations.